In [13]:
#csv 불러오기
!pip install tabulate
import pandas as pd
import numpy as np
from pathlib import Path

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

raw = pd.read_csv("Health_Suppliments_Flipkart_02_04_2022.csv")
print("원본:", raw.shape)
raw.head(3)

원본: (3076, 20)


,category,title,quantity,brand,url,product_id,listing_id,highlights,availability,selling_price,original_price,currency,avg_rating,rating_count,review_count,1_stars_count,2_stars_count,3_stars_count,4_stars_count,5_stars_count
0,Vitamin Supplement,"Revital Women Multivitamin with Calcium, Zinc,...",30 Tablets,Revital,http://dl.flipkart.com/dl/revital-women-multiv...,VSLFUCNZ74TDDANJ,LSTVSLFUCNZ74TDDANJAGY8AW,Multi Vitamin Supplements Tablet Form Suitable...,IN_STOCK,243.0,345.0,INR,4.4,8190,611,211,198,775,2096,4910
1,Vitamin Supplement,MUSCLEBLAZE VITE Multivitamin,90 No,MUSCLEBLAZE,http://dl.flipkart.com/dl/muscleblaze-vite-mul...,VSLFGMHYFZXTEFSW,LSTVSLFGMHYFZXTEFSWCOQLV7,Multi Vitamin Supplements Tablet Form Suitable...,IN_STOCK,455.0,899.0,INR,4.2,41010,3527,2271,1305,3980,10320,23134
2,Vitamin Supplement,"Revital Men Multivitamin with Calcium, Zinc & ...",30 Capsules,Revital,http://dl.flipkart.com/dl/revital-men-multivit...,VSLFUCNZG3X83CGX,LSTVSLFUCNZG3X83CGXYTZXP1,Multi Vitamin Supplements Capsules Form Suitab...,IN_STOCK,214.0,310.0,INR,4.4,5632,513,216,157,491,1281,3487


In [ ]:
#품질검증
def quality_report_full(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    """결측·중복·수치형 이상치(IQR) 비율을 한 번에 진단한다."""
    n_rows = len(df)
    base = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })

    # IQR 이상치 비율
    outlier_pct = {}
    for col in df.select_dtypes(include="number").columns:
        s = df[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier_pct[col] = ((s < lo) | (s > hi)).mean() * 100
    base["outlier_pct_iqr"] = pd.Series(outlier_pct).round(2)

    print(f"[품질 리포트(완전판)] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    return base

In [15]:
# 시나리오 1
qr_flipkart = quality_report_full(raw, "flipkart_raw")
qr_flipkart

[품질 리포트(완전판)] flipkart_raw
  행 수: 3,076  /  열 수: 20
  완전 중복 행: 0건


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr
category,object,0,0.00,5,NaN
title,object,0,0.00,2897,NaN
quantity,object,0,0.00,1246,NaN
brand,object,0,0.00,552,NaN
url,object,0,0.00,3076,NaN
product_id,object,0,0.00,3076,NaN
listing_id,object,0,0.00,3076,NaN
highlights,object,681,22.14,1100,NaN
availability,object,0,0.00,2,NaN
selling_price,float64,0,0.00,1022,8.52


In [16]:
# 시나리오 2 

def f_dedup(df):
    # 판단 근거: product_id는 상품 고유키. 동일 id 중복은 수집 오류로 보고 하나만 남긴다.
    return df.drop_duplicates(subset=["product_id"], keep="first").reset_index(drop=True)

def f_clean_brand(df):
    # 판단 근거: 앞뒤 공백·대소문자 차이는 같은 브랜드로 본다 ('MUSCLEBLAZE' → 'Muscleblaze').
    return df.assign(brand=df["brand"].str.strip().str.title())

def f_parse_quantity(df):
    # 판단 근거: "2 x 30 No" 같은 문자열에서 총 개수를 숫자로 추출한다(묶음 x 개수).
    #            g(그램) 단위 등 개수로 환산 불가한 값은 NaN으로 둔다.
    def to_count(x):
        s = str(x).lower().replace("tablets", "").replace("capsules", "").replace("no", "")
        if "x" in s:
            parts = [p.strip() for p in s.split("x")]
            try:
                return float(parts[0]) * float(parts[1])
            except Exception:
                return np.nan
        nums = "".join(ch for ch in s if (ch.isdigit() or ch == "."))
        try:
            return float(nums)
        except Exception:
            return np.nan
    return df.assign(qty_count=df["quantity"].map(to_count))

def f_split_noreview(df):
    # 판단 근거: avg_rating=0 & rating_count=0은 '별점 0점'이 아니라 '리뷰 없음'이다.
    #            평점 분석을 왜곡하므로 별도 보관 후 본 분석에서 제외한다. (환불 분리와 같은 논리)
    return df[~((df["avg_rating"] == 0) & (df["rating_count"] == 0))].reset_index(drop=True)

def f_fill_highlights(df):
    # 판단 근거: highlights 결측은 상품 설명 누락. KPI에 영향 없으므로 '정보없음'으로 채워 보존한다.
    return df.assign(highlights=df["highlights"].fillna("정보없음"))

# 리뷰 없는 상품은 버리지 않고 따로 보관 (환불 데이터를 별도 보관하던 것과 동일)
noreview = raw[(raw["avg_rating"] == 0) & (raw["rating_count"] == 0)].copy()

flipkart_clean = (
    raw
    .pipe(f_dedup)
    .pipe(f_clean_brand)
    .pipe(f_parse_quantity)
    .pipe(f_split_noreview)
    .pipe(f_fill_highlights)
)

print(f"정제 전: {raw.shape}  →  정제 후: {flipkart_clean.shape}")
print(f"리뷰 없음 보관: {len(noreview)}건")
print(f"qty_count 파싱 실패(g 단위 등): {flipkart_clean['qty_count'].isna().sum()}건")
print(f"brand 종류: {raw['brand'].nunique()} → {flipkart_clean['brand'].nunique()}")
flipkart_clean.head(3)

정제 전: (3076, 20)  →  정제 후: (2423, 21)
리뷰 없음 보관: 653건
qty_count 파싱 실패(g 단위 등): 306건
brand 종류: 552 → 422


,category,title,quantity,brand,url,product_id,listing_id,highlights,availability,selling_price,...,currency,avg_rating,rating_count,review_count,1_stars_count,2_stars_count,3_stars_count,4_stars_count,5_stars_count,qty_count
0,Vitamin Supplement,"Revital Women Multivitamin with Calcium, Zinc,...",30 Tablets,Revital,http://dl.flipkart.com/dl/revital-women-multiv...,VSLFUCNZ74TDDANJ,LSTVSLFUCNZ74TDDANJAGY8AW,Multi Vitamin Supplements Tablet Form Suitable...,IN_STOCK,243.0,...,INR,4.4,8190,611,211,198,775,2096,4910,30.0
1,Vitamin Supplement,MUSCLEBLAZE VITE Multivitamin,90 No,Muscleblaze,http://dl.flipkart.com/dl/muscleblaze-vite-mul...,VSLFGMHYFZXTEFSW,LSTVSLFGMHYFZXTEFSWCOQLV7,Multi Vitamin Supplements Tablet Form Suitable...,IN_STOCK,455.0,...,INR,4.2,41010,3527,2271,1305,3980,10320,23134,90.0
2,Vitamin Supplement,"Revital Men Multivitamin with Calcium, Zinc & ...",30 Capsules,Revital,http://dl.flipkart.com/dl/revital-men-multivit...,VSLFUCNZG3X83CGX,LSTVSLFUCNZG3X83CGXYTZXP1,Multi Vitamin Supplements Capsules Form Suitab...,IN_STOCK,214.0,...,INR,4.4,5632,513,216,157,491,1281,3487,30.0


In [17]:
# 시나리오 3

# 파생변수: 할인율(%)과 5점 리뷰 비율(%)
flipkart_clean = flipkart_clean.assign(
    discount_pct=((flipkart_clean["original_price"] - flipkart_clean["selling_price"])
                  / flipkart_clean["original_price"] * 100).round(1),
    five_star_ratio=(flipkart_clean["5_stars_count"] / flipkart_clean["rating_count"] * 100).round(1),
)

# KPI 1: 카테고리별 — 제품수 / 평균별점 / 평균할인율 / 평균가격
category_kpi = (
    flipkart_clean.groupby("category")
    .agg(n_products=("product_id", "count"),
            avg_rating=("avg_rating", "mean"),
            avg_discount=("discount_pct", "mean"),
            avg_price=("selling_price", "mean"))
    .round(2)
    .sort_values("n_products", ascending=False)
)
print("카테고리별 KPI:")
display(category_kpi)

# KPI 2: 브랜드별 — 제품수 / 평균별점 / 총리뷰수
brand_kpi = (
    flipkart_clean.groupby("brand")
    .agg(n_products=("product_id", "count"),
            avg_rating=("avg_rating", "mean"),
            total_reviews=("review_count", "sum"))
    .round(2)
    .sort_values("total_reviews", ascending=False)
)
print("\n브랜드별 KPI (상위 10):")
display(brand_kpi.head(10))

카테고리별 KPI:


,n_products,avg_rating,avg_discount,avg_price
category,,,,
Vitamin Supplement,826,4.26,47.71,557.65
Protein Supplement,698,4.18,35.74,1672.54
Energy Drinks,476,4.30,32.84,929.48
Milk Drink Mixes,370,4.22,26.48,336.22
Digestive Probiotic,53,4.05,32.77,401.72



브랜드별 KPI (상위 10):


,n_products,avg_rating,total_reviews
brand,,,
Muscleblaze,157,4.19,270219
Optimum Nutrition,21,4.17,118299
Horlicks,43,4.30,116103
Endura,15,3.99,79727
Bigmuscles Nutrition,44,4.12,46905
As-It-Is Nutrition,38,4.07,41445
Cadbury Bournvita,19,4.34,20408
Healthkart,62,4.27,19440
Pediasure,32,4.43,17555


In [18]:
# Part 5 — 저장: Parquet + CSV 용량 비교
import os

category_kpi.to_parquet(OUT_DIR / "flipkart_category_kpi.parquet")
brand_kpi.to_parquet(OUT_DIR / "flipkart_brand_kpi.parquet")

flipkart_clean.to_parquet(OUT_DIR / "flipkart_clean.parquet")
flipkart_clean.to_csv(OUT_DIR / "flipkart_clean.csv", index=False)

p = os.path.getsize(OUT_DIR / "flipkart_clean.parquet")
c = os.path.getsize(OUT_DIR / "flipkart_clean.csv")
print(f"CSV {c:,} bytes  vs  Parquet {p:,} bytes")
print(f"→ Parquet이 CSV보다 약 {c/p:.1f}배 작다 (컬럼 저장 + 압축 + 타입 보존).")

CSV 1,066,414 bytes  vs  Parquet 431,992 bytes
→ Parquet이 CSV보다 약 2.5배 작다 (컬럼 저장 + 압축 + 타입 보존).


In [ ]:
# 결정 로그를 리스트로 누적하다가 마지막에 표로 출력 
decisions = []

def log_decision(step: str, action: str, reason: str, result: str):
    decisions.append({"step": step, "action": action, "reason": reason, "result": result})

log_decision(
    "1. 중복 제거",
    "product_id 기준 중복 drop",
    "product_id는 상품 고유키. 동일 id는 수집 오류로 간주",
    f"{raw.shape[0] - raw['product_id'].nunique()}건 제거",
)
log_decision(
    "2. 문자열 정제",
    "brand를 strip + Title Case",
    "'MUSCLEBLAZE'·'Muscleblaze '는 같은 브랜드",
    f"브랜드 표기 {raw['brand'].nunique()}종 정규화",
)
log_decision(
    "3. 값 파싱",
    "quantity 문자열에서 qty_count 숫자 추출",
    "'2 x 30 No' 등 묶음·단위 혼재를 개수로 통일",
    f"파싱 실패(g 단위 등): {flipkart_clean['qty_count'].isna().sum()}건",
)
log_decision(
    "4. 이상치 분리",
    "avg_rating=0 & rating_count=0 별도 보관",
    "'별점 0점'이 아니라 '리뷰 없음' → 평점 분석 왜곡 방지",
    f"평점 없음 {len(noreview)}건 분리",
)
log_decision(
    "5. 결측치 처리",
    "highlights 결측 '정보없음' 채움",
    "상품 설명 누락은 KPI에 무관 → 행 제거 대신 보존",
    "highlights 결측 0건",
)

decision_log = pd.DataFrame(decisions)
decision_log

,step,action,reason,result
0,1. 중복 제거,product_id 기준 중복 drop,product_id는 상품 고유키. 동일 id는 수집 오류로 간주,0건 제거
1,2. 문자열 정제,brand를 strip + Title Case,'MUSCLEBLAZE'·'Muscleblaze '는 같은 브랜드,브랜드 표기 552종 정규화
2,3. 값 파싱,quantity 문자열에서 qty_count 숫자 추출,'2 x 30 No' 등 묶음·단위 혼재를 개수로 통일,파싱 실패(g 단위 등): 306건
3,4. 이상치 분리,avg_rating=0 & rating_count=0 별도 보관,'별점 0점'이 아니라 '리뷰 없음' → 평점 분석 왜곡 방지,평점 없음 653건 분리
4,5. 결측치 처리,highlights 결측 '정보없음' 채움,상품 설명 누락은 KPI에 무관 → 행 제거 대신 보존,highlights 결측 0건


In [20]:
# 결정 로그를 마크다운 표로도 출력 (README에 붙여넣기 용)
print(decision_log.to_markdown(index=False))

| step           | action                                  | reason                                                | result                      |
|:---------------|:----------------------------------------|:------------------------------------------------------|:----------------------------|
| 1. 중복 제거   | product_id 기준 중복 drop               | product_id는 상품 고유키. 동일 id는 수집 오류로 간주  | 0건 제거                    |
| 2. 문자열 정제 | brand를 strip + Title Case              | 'MUSCLEBLAZE'·'Muscleblaze '는 같은 브랜드            | 브랜드 표기 552종 정규화    |
| 3. 값 파싱     | quantity 문자열에서 qty_count 숫자 추출 | '2 x 30 No' 등 묶음·단위 혼재를 개수로 통일           | 파싱 실패(g 단위 등): 306건 |
| 4. 이상치 분리 | avg_rating=0 & rating_count=0 별도 보관 | '별점 0점'이 아니라 '리뷰 없음' → 평점 분석 왜곡 방지 | 평점 없음 653건 분리        |
| 5. 결측치 처리 | highlights 결측 '정보없음' 채움         | 상품 설명 누락은 KPI에 무관 → 행 제거 대신 보존       | highlights 결측 0건         |


# 📝 종합 프로젝트 보고서


## 1. 데이터 개요
- 출처: Flipkart 건강보조제 상품 데이터 (Kaggle, 2022-04-02 수집, 통화 INR)
- 규모: 원본 3,076행 → 정제 후 2,423행, 21개 컬럼(파생 2개 포함)
- 카테고리: 비타민·단백질·밀크드링크·에너지드링크·유산균 5종
- 선택이유: 약국에서 아르바이트를 오랫동안 해왔기 때문에 약에 관한 관심이 있어서 건강보조제 데이터를 가져왔습니다.

## 2. 발견한 품질 문제
- 타입/문자열 혼재: quantity가 `"2 x 30 No"`, `"250 g"` 등 단위·묶음 뒤섞임
- 숨은 결측: avg_rating=0 & rating_count=0인 '리뷰 없음' 상품 653건 (0점으로 오인 위험)
- 결측: highlights 681건
- 표기 혼재: 브랜드명 대소문자·공백 불일치(552종)

## 3. 처리 결정과 근거
1. product_id 중복 → 고유키 기준 제거
2. brand → strip + Title Case로 통일
3. quantity → qty_count(숫자)로 파싱, g 단위 등은 NaN 유지
4. avg_rating=0 & rating_count=0 → '리뷰 없음'으로 별도 보관, 본 분석 제외
5. highlights 결측 → '정보없음'으로 보존

## 4. 주요 KPI 결과
- (카테고리 관점) 제품 수 1위: Vitamin Supplement
- (브랜드 관점) 총 리뷰수 1위: Muscleblaze

## 5. 한계와 후속 작업
- 날짜가 없어 시계열(계절성) 분석 불가 → 리뷰 작성일 있는 데이터와 병합 시 가능
- qty_count 파싱 실패(g 단위 등) 306건 → 단위 표준화 규칙 추가 필요
- 인도(INR) 시장 데이터 → 한국 약국과 직접 비교엔 한계, 카테고리 구조 참고용
- (후속) 이 KPI를 Plotly로 시각화 